# 3. Multi-Head Attention

Multi-head attention runs multiple attention mechanisms in parallel, each with different learned projections.
This allows the model to attend to different types of information simultaneously!


In [1]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np


## Why Multiple Heads?

Single attention learns one type of relationship. Multi-head attention learns multiple types:
- Head 1 might focus on syntactic relationships
- Head 2 might focus on semantic relationships
- Head 3 might focus on long-range dependencies
- etc.


In [2]:
# Multi-head attention splits d_model into num_heads smaller subspaces
d_model = 64
num_heads = 8
d_k = d_model // num_heads  # Dimension per head (64/8 = 8)

print(f"d_model: {d_model}")
print(f"num_heads: {num_heads}")
print(f"d_k (dimension per head): {d_k}")
print(f"\nEach head operates in a {d_k}-dimensional subspace!")


d_model: 64
num_heads: 8
d_k (dimension per head): 8

Each head operates in a 8-dimensional subspace!


## Multi-Head Attention Implementation

1. Split Q, K, V into multiple heads
2. Apply attention to each head independently
3. Concatenate all heads
4. Apply output projection


In [ ]:
class MultiHeadAttention(nn.Module):
    """Multi-head attention from scratch"""
    
    def __init__(self, d_model, num_heads):
        # inheriting pytorch NN libraries
        super().__init__()
        # ensuring the d_model must be divisible by num_heads
        assert d_model % num_heads == 0, "d_model must be divisible by num_heads"
        
        # defining the dimension_model (d_model)
        self.d_model = d_model
        # defining the number_heads (num_heads)
        self.num_heads = num_heads
        # calculating dimension_key (d_k)
        #   by d_model per head
        self.d_k = d_model // num_heads # floor division - remove the decimal point and rounding up
        
        # Linear projections for Q, K, V
        # Utilize linear projection for Q, K and V
        # Dimension model and sequence length are equal
        #   Q (Query)
        self.W_q = nn.Linear(d_model, d_model, bias=False)
        #   K (Key) 
        self.W_k = nn.Linear(d_model, d_model, bias=False)
        #   V (Value)
        self.W_v = nn.Linear(d_model, d_model, bias=False)
        
        # Output projection
        # Utilize one linear projection output
        self.W_o = nn.Linear(d_model, d_model)
        
    def forward(self, x):
        """
        Args:
            x: [batch_size, seq_len, d_model]
            # x (input from word embedding - contains arguement of
                    batch_size - conversations (number of sequences processed at once)
                    seq_len - tokens of words (tokens per words)
                    d_model - dimension of model for x (total embedding dimension for each token))
        Returns:
            output: [batch_size, seq_len, d_model]
            # ouput (the results from transformer - contains the similar argument as x)
        """
        # defining these 3 argument to initialize the shape/dimension of x input
        batch_size, seq_len, d_model = x.shape
        
        # Create Q, K, V
        # Transform the x input into Q,K and V projection
        Q = self.W_q(x)  # [batch_size, seq_len, d_model]
        K = self.W_k(x)  # [batch_size, seq_len, d_model]
        V = self.W_v(x)  # [batch_size, seq_len, d_model]
        
        # Reshape for multi-head: [batch_size, num_heads, seq_len, d_k]
        # Original shape for self-attention (batch_size, seq_len, d_model) as overall
        # Now, the Q,K and V will be split based on number heads
        # Multi-head attention (batch_size, seq_len, num_heads(additional arg), d_k) 
        #   split into multi-heads
        #   dimension of multi-heads (d_k) based on = d_model // num_heads
        # These Q,K and V are transposed to be ready in matmul
        #   multi-head dimenstion [batch_size, seq_len, num_heads, d_k]
        #   transposed multi-head dimension [batch_size, num_heads, seq_len, d_k]   
        #       swap dimension between dim1(seq_len) and dim2 (num_heads)
        # Multiplication will deal with last 2 dimensions (num_heads, d_k) and (seq_len, d_k) 
        Q = Q.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        K = K.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        V = V.view(batch_size, seq_len, self.num_heads, self.d_k).transpose(1, 2)
        
        print("Multi-head attention dimension, spliced, squeezed together and transposed")
        print(f"Q (Query): {Q.shape}")
        print(f"K (Key): {K.shape}")
        print(f"V (Value): {V.shape}")
        print()

        # Calculation done similar with self-attention
        # Scaled dot-product attention for each head
        # Get the attention score (QK^T/ sqrt(d_k)) (1st and 2nd)
        scores = torch.matmul(Q, K.transpose(-2, -1)) / np.sqrt(self.d_k)
        print("After calculation attention_score shape (QK^T/ sqrt(d_k))")
        print("(batch_size, num_heads, seq_len, seq_len)")
        print(f"{scores.shape}")
        print()

        # Get the softmax() of attention_score (3rd)
        #   to get aattention_weights       
        attention_weights = F.softmax(scores, dim=-1)
        print("After calculation attention_weight shape softmax((QK^T/ sqrt(d_k), dim=-1)")
        print("(batch_size, num_heads, seq_len, seq_len)")
        print(f"{attention_weights.shape}")
        print()

        # calculate the output (1st and 2nd calc) x V
        attention_output = torch.matmul(attention_weights, V)  # [batch_size, num_heads, seq_len, d_k]
        
        # Concatenate heads: [batch_size, seq_len, d_model]
        # contiguous() is used in re-arrange data inside memory before 
        #   executing view() for re-shaping dimension
        attention_output = attention_output.transpose(1, 2).contiguous()
        # From multi heads attention [batch_size, seq_len, num_heads, d_k] into
        # Reshape back into original [batch_size, seq_len, d_model]
        #   to concatenate everything from multi-head into overall as one
        attention_output = attention_output.view(batch_size, seq_len, d_model)
        
        # Output projection
        output = self.W_o(attention_output)
        
        return output

# Test multi-head attention
batch_size = 2
seq_len = 10
num_heads = 8
d_model = 64

mha = MultiHeadAttention(d_model, num_heads)
x = torch.randn(batch_size, seq_len, d_model)


print(f"Input shape (batch_size, seq_len, d_model):\n{x.shape}")
print()
print("Properties of linear projection in Multi-head attention (dim_model, dim_model, bias_flag):")
print(f"Q (Query): {mha.W_q}")
print(f"K (Key): {mha.W_k}")
print(f"V (Value): {mha.W_v}")
print(f"O (Output - results): {mha.W_o}")
print()

print("Splice Q,K and V in multi-head (batch_size, seq_len, num_heads, d_k):")
# Calculation of multi head atttention put here, to print out input and linear fx before
#   mult-head attenstion dimensions
output = mha(x) # <-- actually this calls mha.forward(x)

print(f"Output shape (batch_size, seq_len, d_model):\n{output.shape}")
print(f"\nMulti-head attention successfully processes {num_heads} attention heads in parallel!")


Input shape (batch_size, seq_len, d_model):
torch.Size([2, 10, 64])

Properties of linear projection in Multi-head attention (dim_model, dim_model, bias_flag):
Q (Query): Linear(in_features=64, out_features=64, bias=False)
K (Key): Linear(in_features=64, out_features=64, bias=False)
V (Value): Linear(in_features=64, out_features=64, bias=False)
O (Output - results): Linear(in_features=64, out_features=64, bias=True)

Splice Q,K and V in multi-head (batch_size, seq_len, num_heads, d_k):
Multi-head attention dimension, spliced, squeezed together and transposed
Q (Query): torch.Size([2, 8, 10, 8])
K (Key): torch.Size([2, 8, 10, 8])
V (Value): torch.Size([2, 8, 10, 8])

After calculation attention_score shape (QK^T/ sqrt(d_k))
(batch_size, num_heads, seq_len, seq_len)
torch.Size([2, 8, 10, 10])

After calculation attention_weight shape softmax((QK^T/ sqrt(d_k), dim=-1)
(batch_size, num_heads, seq_len, seq_len)
torch.Size([2, 8, 10, 10])

Output shape (batch_size, seq_len, d_model):
torch.S